# Attention from Scratch

## Important Information

- Email: [joanna_bieri@redlands.edu](mailto:joanna_bieri@redlands.edu)
- Office Hours take place in Duke 209, [Office Hours Schedule](https://joannabieri.com/schedule.html)
- [Class Website](https://joannabieri.com/machine_learning.html)
- [Syllabus](https://joannabieri.com/machinelearning/IntroMachineLearning.pdf)

:::{.callout-important icon=false}
## How to use these notes

Two kinds of box show up in these notes.

**Blue Q boxes** are questions for you to answer **by hand, in a notebook, with a pen.** Not because I am old fashioned. Writing something down by hand is slow, and slow is the point: it is very hard to write an explanation you do not actually understand. You are welcome to use AI in this class for the mechanics of code, but these boxes are the part where you do the thinking yourself. Bring your written notes to class, I will ask to see them.

**Green You Try boxes** are optional code for you to work through. Nothing is collected and nothing is graded. They are there because you will learn more from changing a number and rerunning than from watching me do it.

**Every code cell begins with a tag** that tells you what to do with it.

- `# RUN THIS.` Setup, loading data, a plot. Copy it, run it, move on. You do not need to be able to write it from memory.
- `# LEARN TO WRITE THIS.` The pattern of the day. The homework will ask you for it, and so will the exam. Type it out yourself at least once rather than pasting it.
- `# DEMO ONLY.` Code that exists to show one idea. You do not need to be able to write it yourself.
- `# UNDERSTAND THESE STEPS.` Code that shows how something works inside, one small step at a time. Read it and make sure you understand what each line does. You do not need to memorize it or be able to write it from scratch.
:::

**Reading:** Geron chapter 14, the section **Attention Mechanisms**. Its code adds attention to the translation model from the section just before it (An Encoder-Decoder Network for Neural Machine Translation), so skim that one if the code looks confusing. Focus on the `attention()` function, it is the same thing we write today.

When we built RNNs, the GRU learned our long made up reviews! But at the end of the notes we saw two problems:

1. **Everything has to fit in one memory.** By the last word the whole review has to be squeezed into 32 numbers.
2. **It reads one word at a time.** To get from word 1 to the end, the information (and the gradient on the way back) has to go through 99 steps.

**Attention** fixes both of these. Instead of passing a memory from one word to the next, every word gets to **look at every other word directly**, in one step.

![](images/02-rnn-vs-attention.png)

A **transformer** is a kind of neural network built out of attention layers stacked on top of each other, with some ordinary `nn.Linear` layers in between. It was introduced in 2017 in a paper called *Attention Is All You Need*. Before transformers, language models were mostly RNNs like the ones we already built.

Attention is the main idea inside every transformer, and every LLM you have used (ChatGPT, Claude, Gemini, ...) is a stack of transformer blocks. So today is a big day! We are going to build attention by hand, one step at a time. Next we put it inside a transformer block.

Here is the plan for today:

1. The idea: queries, keys and values.
2. Attention for one word, by hand, on a 4 word sentence.
3. Every word at once, with matrices.
4. Self-attention: the network learns its own queries, keys and values.
5. Attention on the long made up reviews we used for the RNN, and a look inside at what it learned.

---

# The Idea - Queries, Keys and Values

Take the sentence **"the pizza was cold"**. To understand the word "cold" you need to know **what** is cold. So "cold" should go look at the other words and pull in information from "pizza".

Attention does this with three lists of numbers for every word:

- a **query**: what this word is looking for. For "cold": what thing am I describing?
- a **key**: what this word has to offer, the thing a query gets compared to.
- a **value**: the information this word hands over if somebody looks at it.

The names come from searching on the internet. You type a query, it gets compared to the key (eg. the title) of every page, and you get back the values (the pages) that matched best. The difference is that attention does not just pick one page. Instead it takes **a weighted average of all the values**, with bigger weights for better matches.

![](images/01-query-key-value.png)

---

# Attention for One Word, by Hand

Let's do one by hand! First we need a query, a key and a value for every word.

**Where do these numbers come from?** In a real network nobody picks them, the network learns them (that is the Self-Attention section below). Today I made them up by hand, and I gave every spot a meaning so we can follow what is happening. In a trained network the numbers do not come with labels we can read or understand in the way we will in the example.

**Why 4 numbers and 2 numbers?** The sizes are a choice, like the number of neurons in a layer. The query and the key have to be the SAME length, because we are going to compare them with a dot product. The value can be any length, it is the information that gets handed over. I picked small sizes so we can do the math by hand. Real models use 64 or more.

Here is what each spot means in our made up example:

- **key** (what the word offers): [is a thing, is food, is a describing word, is a little grammar word]
- **value** (what the word hands over): [about food, about temperature]

| word | key | value |
|---|---|---|
| the | [0, 0, 0, 1] | [0, 0] |
| pizza | [1, 1, 0, 0] | [1, 0] |
| was | [0, 0, 0, 1] | [0, 0] |
| cold | [0, 0, 1, 0] | [0, 1] |

So "pizza" offers "I am a thing, and I am food", and it hands over "I am about food". "cold" offers "I am a describing word", and it hands over "I am about temperature".

"cold" is a describing word, so it needs to find the thing it is describing. Its query asks for a thing: **[4, 0, 0, 0]**. (The 4 says how strongly it is asking.)

**Step 1. Compare the query to every key with a dot product.** We need a number that says how well each word matches what "cold" is looking for. The dot product multiplies the numbers in the same spot and adds them up, so it is big when the query and the key have big numbers in the SAME spots.

- the: $4 \cdot 0 + 0 \cdot 0 + 0 \cdot 0 + 0 \cdot 1 = 0$
- pizza: $4 \cdot 1 + 0 \cdot 1 + 0 \cdot 0 + 0 \cdot 0 = 4$
- was: $0$
- cold: $0$

Only "pizza" is a thing, so only "pizza" matches.

**Step 2. Divide by $\sqrt{d}$**, where $d$ is how many numbers are in a key. This keeps the scores from getting too big when the keys are long (more on this below). Here $d = 4$, so we divide by 2. The scores are now **0, 2, 0, 0**.

**Step 3. Softmax**, the same softmax we used for the 10 clothing classes with CNNs. We want to take an average, so we need weights that are all positive and add up to 1, and the best match should get the biggest weight. Take $e$ to each score and divide by the total.

$$e^0 = 1, \quad e^2 = 7.389, \quad e^0 = 1, \quad e^0 = 1, \qquad \text{total} = 10.389$$

$$\text{weights} = \frac{1}{10.389}, \frac{7.389}{10.389}, \frac{1}{10.389}, \frac{1}{10.389} = 0.096, \ 0.711, \ 0.096, \ 0.096$$

**Step 4. Take the weighted average of the values.** This is the whole point! The new vector for "cold" is a mix of the information from the words it paid attention to, mostly from "pizza".

$$0.096 \cdot [0, 0] + 0.711 \cdot [1, 0] + 0.096 \cdot [0, 0] + 0.096 \cdot [0, 1] = [0.711, \ 0.096]$$

Look at what happened! The word "cold" started out as only about temperature, [0, 1]. After attention its new vector is [0.711, 0.096], mostly about food. It now carries the information "the thing that is cold is food". This new vector is what gets sent on to the next layer of the network.

NOTE: "cold" did not look at itself very much, so its new vector lost most of its own temperature information. In a transformer the word's original vector gets added back after attention, so nothing is lost. We will see this when we build a transformer block.

Here are the same four steps in PyTorch, so we can check our numbers.

In [ ]:
# RUN THIS. Imports.
import math
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

In [ ]:
# UNDERSTAND THESE STEPS. Attention for the word "cold", one step at a time.
words = ["the", "pizza", "was", "cold"]
K = torch.tensor([[0., 0, 0, 1],      # key for "the":   a little grammar word
                  [1, 1, 0, 0],       # key for "pizza": a thing, and food
                  [0, 0, 0, 1],       # key for "was":   a little grammar word
                  [0, 0, 1, 0]])      # key for "cold":  a describing word
V = torch.tensor([[0., 0],            # value for "the"
                  [1, 0],             # value for "pizza": about food
                  [0, 0],             # value for "was"
                  [0, 1]])            # value for "cold":  about temperature
q_cold = torch.tensor([4., 0, 0, 0])  # the query for "cold": looking for a thing

scores = K @ q_cold                   # Step 1: one dot product per word (@ multiplies the matrix K by the vector q_cold)
print("scores:        ", scores)
scaled = scores / math.sqrt(4)        # Step 2: divide by the square root of d = 4
print("scaled scores: ", scaled)
weights = torch.softmax(scaled, dim=0)  # Step 3: softmax, the weights add up to 1
print("weights:       ", weights)
new_cold = weights @ V                # Step 4: the weighted average of the values
print("new vector for cold:", new_cold)

:::{.callout-note icon=false}
## Q1. Write this one out by hand

**a.** In the "cold" example, which word got the most weight? Why does that make sense for the word "cold"?

**b.** Why do we put the scores through a softmax, instead of using the scores 0, 2, 0, 0 as the weights?
:::

---

# Every Word at Once - The Attention Formula

When we are doing this process, every word gets its own query, so we do the same four steps for every word at the same time. First we stack the queries as the rows of a matrix $Q$, the keys as the rows of $K$ and the values as the rows of $V$. Then the dot product of every query with every key is one matrix multiplication, $Q K^T$. Row $i$, column $j$ of $QK^T$ is how well word $i$'s query matches word $j$'s key.

![](images/03-attention-matrices.png)

Here is the whole thing in one line. This is the formula from the paper that started transformers (Attention Is All You Need, 2017):

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{Q K^T}{\sqrt{d}}\right) V$$

with the softmax done on each row. Each row of the weights adds up to 1, and each row of the output is the new vector for one word.

Now let's give every word a query. "pizza" is a thing, so it looks for a describing word (what describes me?). Its query has a 4 in the describing word spot. "the" and "was" have queries of all zeros, they are not looking for anything. "cold" keeps the query from before, [4, 0, 0, 0].

:::{.callout-note collapse="true" icon=false}
## The matrix math, step by step (click to open)

If it has been a while since linear algebra, here is the same calculation for our 4 words, written out with the matrices.

**The transpose.** $K^T$ (say "K transpose") is $K$ flipped so the rows become the columns. $K$ has one row per word, so $K^T$ has one **column** per word.

$$
K = \begin{bmatrix} 0 & 0 & 0 & 1 \\ 1 & 1 & 0 & 0 \\ 0 & 0 & 0 & 1 \\ 0 & 0 & 1 & 0 \end{bmatrix}
\begin{matrix} \leftarrow \text{the} \\ \leftarrow \text{pizza} \\ \leftarrow \text{was} \\ \leftarrow \text{cold} \end{matrix}
\qquad
K^T = \begin{bmatrix} 0 & 1 & 0 & 0 \\ 0 & 1 & 0 & 0 \\ 0 & 0 & 0 & 1 \\ 1 & 0 & 1 & 0 \end{bmatrix}
$$

The columns of $K^T$ are the keys of the, pizza, was, cold, in that order.

**Multiply $Q K^T$.** When you multiply two matrices, the number in row $i$, column $j$ of the answer is the dot product of row $i$ of the first matrix with column $j$ of the second. Row $i$ of $Q$ is word $i$'s query, and column $j$ of $K^T$ is word $j$'s key. So $QK^T$ holds ALL of the query-key dot products at once, the same dot products we did by hand.

$$
Q = \begin{bmatrix} 0 & 0 & 0 & 0 \\ 0 & 0 & 4 & 0 \\ 0 & 0 & 0 & 0 \\ 4 & 0 & 0 & 0 \end{bmatrix}
\begin{matrix} \leftarrow \text{the} \\ \leftarrow \text{pizza} \\ \leftarrow \text{was} \\ \leftarrow \text{cold} \end{matrix}
\qquad
Q K^T = \begin{bmatrix} 0 & 0 & 0 & 0 \\ 0 & 0 & 0 & 4 \\ 0 & 0 & 0 & 0 \\ 0 & 4 & 0 & 0 \end{bmatrix}
$$

Look at the last row, for "cold": [0, 4, 0, 0]. Those are exactly the scores from the by hand example! The second row says "pizza" matches "cold" (the 4 in the last column).

**Divide by $\sqrt{4} = 2$ and softmax each row.** Every row becomes a set of weights that adds up to 1.

$$
\frac{Q K^T}{2} = \begin{bmatrix} 0 & 0 & 0 & 0 \\ 0 & 0 & 0 & 2 \\ 0 & 0 & 0 & 0 \\ 0 & 2 & 0 & 0 \end{bmatrix}
\quad \xrightarrow{\text{softmax each row}} \quad
\text{weights} = \begin{bmatrix} 0.25 & 0.25 & 0.25 & 0.25 \\ 0.096 & 0.096 & 0.096 & 0.711 \\ 0.25 & 0.25 & 0.25 & 0.25 \\ 0.096 & 0.711 & 0.096 & 0.096 \end{bmatrix}
$$

**Multiply by $V$.** Row $i$ of the answer is word $i$'s weights times the values, which is the weighted average from Step 4.

$$
\text{weights} \; V = \begin{bmatrix} 0.25 & 0.25 & 0.25 & 0.25 \\ 0.096 & 0.096 & 0.096 & 0.711 \\ 0.25 & 0.25 & 0.25 & 0.25 \\ 0.096 & 0.711 & 0.096 & 0.096 \end{bmatrix}
\begin{bmatrix} 0 & 0 \\ 1 & 0 \\ 0 & 0 \\ 0 & 1 \end{bmatrix}
= \begin{bmatrix} 0.25 & 0.25 \\ 0.096 & 0.711 \\ 0.25 & 0.25 \\ 0.711 & 0.096 \end{bmatrix}
$$

The last row is [0.711, 0.096], the new "cold" from before. Why bother with matrices? One matrix multiplication does all 16 dot products at once, and computers (especially GPUs) are REALLY fast at matrix multiplication. A real sentence with 1,000 words needs a million dot products, so this matters.
:::

In PyTorch `@` is matrix multiplication and `K.T` is the transpose. Here is the same thing in code.

In [ ]:
# UNDERSTAND THESE STEPS. All four words at once.
Q = torch.tensor([[0., 0, 0, 0],      # query for "the": looking for nothing
                  [0, 0, 4, 0],       # query for "pizza": looking for a describing word
                  [0, 0, 0, 0],       # query for "was": looking for nothing
                  [4, 0, 0, 0]])      # query for "cold": looking for a thing, same as before

scores = Q @ K.T / math.sqrt(4)              # K.T flips K so the keys are the columns: (4 words, 4 words)
weights = torch.softmax(scores, dim=1)       # dim=1: softmax along each row
output = weights @ V                         # (4 words, 2 numbers)

print("weights:")
print(weights.round(decimals=3))
print("output:")
print(output.round(decimals=3))

In [ ]:
# RUN THIS. The weights as a heat map. Row = the word that is looking, column = the word it looks at.
def show_weights(weights, words, title):
    plt.figure(figsize=(5, 4))
    plt.imshow(weights, cmap="Blues", vmin=0, vmax=1)
    plt.xticks(range(len(words)), words, rotation=90)     # rotation=90: write the words up and down so they do not overlap
    plt.yticks(range(len(words)), words)
    plt.xlabel("looks at (key)")
    plt.ylabel("word that is looking (query)")
    plt.colorbar()
    plt.title(title)
    plt.show()

show_weights(weights, words, "Attention weights, the pizza was cold")

Read the heat map one row at a time. The "cold" row is the one we did by hand, most of its weight is on "pizza". The "pizza" row looks most at "cold" (0.711), so the new vector for "pizza" is mostly about temperature: the pizza is the thing that is cold. The rows for "the" and "was" are all 0.25: a query of all zeros matches every key the same, so the weight is spread evenly.

NOTE: PyTorch has this formula built in, `torch.nn.functional.scaled_dot_product_attention`. It gives the same output and it is much faster, so this is what real transformers use. We write our own today so we can see every step.

In [ ]:
# RUN THIS. The built in version gives the same output.
import torch.nn.functional as F
print(F.scaled_dot_product_attention(Q, K, V).round(decimals=3))

## Why divide by $\sqrt{d}$?

In a real model the queries and keys have 64 or more numbers, not 4. A dot product adds up one product for every number, so with more numbers the scores get bigger. Big scores make the softmax put almost all the weight on one word. Look at what happens to two scores when they are 10 times bigger:

| scores | softmax weights |
|---|---|
| 2, 1 | 0.731, 0.269 |
| 20, 10 | 0.99995, 0.00005 |

In the second row almost all the weight is on one word, so it is not really an average anymore. When the weights are stuck near 0 and 1 the gradients through the softmax are close to 0, so training gets SLOW. Dividing by $\sqrt{d}$ keeps the scores a reasonable size no matter how many numbers are in a key.

Why the square root, and not just $d$? If the numbers in the queries and keys are random-ish, the size of a dot product of $d$ numbers grows like $\sqrt{d}$, not like $d$ (positive and negative products partly cancel out). So dividing by $\sqrt{d}$ is exactly enough to bring the scores back to about the same size. In our pizza example $d = 4$, so the score 4 became 2.

:::{.callout-note icon=false}
## Q2. Write this one out by hand

**a.** A sentence has 10 words. What is the shape of the weights matrix? What does row 3 of it tell you?

**b.** Why do we divide the scores by $\sqrt{d}$? What goes wrong if we do not?
:::

---

# Self-Attention - The Network Learns Q, K and V

So far we made up the queries, keys and values by hand. In a real network we do not write them, the network learns them! Each word starts as its embedding (from `nn.Embedding`), and the network makes the query, the key and the value from the embedding with **three `nn.Linear` layers**. Those weights are learned by gradient descent like every other weight. So the network learns what each word should look for and what each word should offer. For example, the query layer could learn to turn the embedding of "cold" into something like our [4, 0, 0, 0], "I am looking for a thing". In a trained network the numbers will not have names like ours, but they do the same job.

This is called **self-attention**, because the words of a sentence are looking at the other words of the **same** sentence.

Like `ReviewNet` for the RNN, attention does not fit in `nn.Sequential`, so we write it as a class. The `forward` function is the formula from above, line by line. It gives back the new vectors AND the weights, so we can look at the weights later.

**Go through the code below slowly and say in your own words what is happening and the order of what is happening. Note that this is JUST the attention layer**

In [ ]:
# LEARN TO WRITE THIS. Self-attention as a layer.
class SelfAttention(nn.Module):

    def __init__(self, d):                          # d = how many numbers each word has
        super().__init__()                          # always the first line
        self.query = nn.Linear(d, d, bias=False)    # makes a query from each word's d numbers
        self.key = nn.Linear(d, d, bias=False)      # makes a key
        self.value = nn.Linear(d, d, bias=False)    # makes a value

    def forward(self, X):                           # X is (reviews, words, d)
        Q = self.query(X)                           # (reviews, words, d)
        K = self.key(X)                             # (reviews, words, d)
        V = self.value(X)                           # (reviews, words, d)
        scores = Q @ K.transpose(1, 2)              # every query dotted with every key: (reviews, words, words)
        scores = scores / math.sqrt(K.shape[-1])    # divide by the square root of d
        weights = torch.softmax(scores, dim=-1)     # dim=-1: softmax along the last dimension, each row
        return weights @ V, weights                 # new vectors (reviews, words, d) and the weights

There are two new things in that code:

1. **`K.transpose(1, 2)` instead of `K.T`.** In our example, we send in a whole batch of reviews at once, so `K` has three sizes: (reviews, words, d). Think of it as a stack of K matrices, one for each review. We want to flip each review's K, which means swapping size 1 (words) and size 2 (d) and leaving size 0 (the reviews) alone. That is what `transpose(1, 2)` does. `K.T` would flip ALL the sizes, reviews included, and mix the reviews up.
2. **`bias=False`.** An `nn.Linear` layer normally does weights times the input, plus a bias (an intercept, like the $b$ in $y = mx + b$). The paper that started transformers writes the queries, keys and values as just a weight matrix times the word's numbers, so we turn the bias off. (It would also work with a bias.)

Let's send a fake batch through and check the shapes.

In [ ]:
# DEMO ONLY. A fake batch: 2 reviews, 5 words each, 8 numbers per word.
torch.manual_seed(42)
attention = SelfAttention(8)
X_fake = torch.randn(2, 5, 8)

new_vectors, weights = attention(X_fake)
print("going in:   ", X_fake.shape)
print("weights:    ", weights.shape)
print("coming out: ", new_vectors.shape)
print('--------------------')
print("each row of weights adds up to:\n", weights[0].sum(dim=1))

Let's walk through where each shape comes from:

- **Going in, (2, 5, 8):** 2 reviews, 5 words in each review, and every word is 8 numbers (its embedding).
- **Q, K and V, each (2, 5, 8):** each `nn.Linear(8, 8)` turns a word's 8 numbers into 8 new numbers, so every word gets a query, a key and a value with 8 numbers each.
- **The weights, (2, 5, 5):** for each review, every one of the 5 words (the rows) gets a weight for every one of the 5 words (the columns). This is the 5 by 5 heat map, one for each review. Each row adds up to 1, and that is what the last print checks.
- **Coming out, (2, 5, 8):** each word's new vector is a weighted average of the 5 values, and every value has 8 numbers, so every word still has 8 numbers.

So the same shape comes out as went in: one new vector for every word. This is important! It means we can put the output of one attention layer into another attention layer, and stacking layers like this is exactly what a transformer does.

:::{.callout-note icon=false}
## Q3. Write this one out by hand

**a.** In self-attention, where do the queries, keys and values come from? Who picks the numbers in them?

**b.** How many numbers does `SelfAttention(8)` have to learn? Does that change if the reviews are 1,000 words long?
:::

---

# Attention on the Long Reviews

Okay, now let's go back to the made up reviews we used for the RNN and see if attention can do what the RNN could not! These are random filler words with **one feeling word** at a random spot, and the label is positive or negative. The plain RNN got 0.493 on the 100 word reviews (a coin flip) and the GRU got 1.0 after 3 epochs. Below is the same data and the same training function, so we can compare.

In [ ]:
# RUN THIS. The made up reviews and the training function from the RNN notes.
feeling_words = ["great", "good", "fun", "loved", "wonderful",       # positive, ids 0 to 4
                 "bad", "boring", "awful", "hated", "terrible"]      # negative, ids 5 to 9
filler_words = ["the", "movie", "was", "and", "a", "plot", "actors", "it", "story", "long",
                "music", "ending", "i", "saw", "with", "my", "friends", "on", "friday", "popcorn"]
vocab = feeling_words + filler_words
vocab_size = len(vocab)

def make_reviews(n_reviews, n_words, seed):
    generator = torch.Generator().manual_seed(seed)       # so we get the same reviews every time
    X = torch.randint(10, vocab_size, (n_reviews, n_words), generator=generator)
    feeling = torch.randint(0, 10, (n_reviews,), generator=generator)
    spot = torch.randint(0, n_words, (n_reviews,), generator=generator)
    X[torch.arange(n_reviews), spot] = feeling
    y = (feeling < 5).long()
    return X, y

def train_model(model, X_train, y_train, X_valid, y_valid, n_epochs):
    torch.manual_seed(42)                         # so the batches come in the same order every time
    train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=32, shuffle=True)
    loss_fn = nn.CrossEntropyLoss()               # the loss for picking a class
    optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
    valid_accuracy = []
    for epoch in range(n_epochs):
        model.train()                             # training mode
        for X_batch, y_batch in train_loader:
            optimizer.zero_grad()                 # clear the old gradients
            loss = loss_fn(model(X_batch), y_batch)   # forward pass and the loss
            loss.backward()                       # backward pass
            optimizer.step()                      # one step of Adam
        model.eval()                              # prediction mode
        with torch.no_grad():
            y_pred = model(X_valid).argmax(dim=1)     # the class with the biggest score
            accuracy = (y_pred == y_valid).float().mean().item()
        valid_accuracy.append(accuracy)
        print("epoch", epoch + 1, "  valid accuracy:", round(accuracy, 4))
    return valid_accuracy

X_train_long, y_train_long = make_reviews(4000, n_words=100, seed=0)
X_valid_long, y_valid_long = make_reviews(1000, n_words=100, seed=1)
print("X_train_long:", X_train_long.shape)

Our network is the `ReviewNet` from the RNN notes, but we take out the RNN and put in attention:

1. `nn.Embedding` turns every word into 8 numbers. (8 is a choice, the same size we used for the RNN.)
2. `SelfAttention` gives every word a new vector, made from the words it looked at. **This is the class we build above!!**
3. We need one summary for the whole review, so we **average** the new vectors over the words with `.mean(dim=1)`. (The RNN used its last memory for this.) Averaging works here because attention already mixed the important information into every word, so the average still has it.
4. `nn.Linear` turns the summary into 2 scores, negative and positive.

In [ ]:
# LEARN TO WRITE THIS. A review classifier with attention instead of an RNN.
class ReviewAttention(nn.Module):

    def __init__(self):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, 8)     # every word becomes 8 numbers
        self.attention = SelfAttention(8)            # every word looks at every word
        self.out = nn.Linear(8, 2)                   # 2 scores: negative, positive

    def forward(self, X):
        vectors = self.embed(X)                      # (reviews, words, 8)
        new_vectors, weights = self.attention(vectors)   # TWO things come back, here we only need new_vectors
        summary = new_vectors.mean(dim=1)            # average over the words: (reviews, 8)
        scores = self.out(summary)
        return scores

torch.manual_seed(42)
attention_model = ReviewAttention()
accuracy_attention = train_model(attention_model, X_train_long, y_train_long, X_valid_long, y_valid_long, n_epochs=5)

In [ ]:
# RUN THIS. How many numbers does it have to learn?
n_params = 0
for p in attention_model.parameters():
    n_params = n_params + p.numel()
print("attention model:", n_params)

WOW! **1.0 after the first epoch**, on the same 100 word reviews where the RNN never did better than a coin flip. And it only has 450 numbers to learn (the GRU had 4,338).

Why does it learn so easily? Look back at the picture at the top. In the RNN, the feeling word at spot 3 has to pass through 97 steps to reach the end, and the gradient has to come back through all 97. With attention, every word is **one step** from every other word, no matter how far apart they are in the review. We can check this the same way we did for the RNN, by measuring the gradient for every word in a brand new model.

In [ ]:
# DEMO ONLY. The size of the gradient for each word, in a new attention model (compare the RNN).
torch.manual_seed(42)
new_model = ReviewAttention()

vectors = new_model.embed(X_train_long[:500])
vectors.retain_grad()                                  # keep the gradient for these
new_vectors, weights = new_model.attention(vectors)
scores = new_model.out(new_vectors.mean(dim=1))
loss = nn.CrossEntropyLoss()(scores, y_train_long[:500])
loss.backward()

gradient_size = vectors.grad.norm(dim=2).mean(dim=0)   # one number per word
print("last word:  ", gradient_size[99].item())
print("word 50:    ", gradient_size[49].item())
print("first word: ", gradient_size[0].item())

The first word gets the same size of gradient as the last word, about 0.0000035. That number is small because this is a brand new model, what matters is that it is the SAME for every word. The RNN gave the last word about 0.00014 and the first 25 words exactly 0. With attention the gradient does not have to go back through 99 steps, so it does not get smaller and smaller. A feeling word at the start of a review counts as much as one at the end.

:::{.callout-note icon=false}
## Q4. Write this one out by hand

**a.** Use the picture at the top of the notes to explain why the RNN failed on the 100 word reviews and attention did not.

**b.** Attention gets every word the same size of gradient. Why does that matter when the feeling word can be anywhere in the review?
:::

---

# Looking Inside - What Did It Learn to Look At?

One really nice thing about attention is that we can look at the weights and see which words each word looked at. 

Let's make up one short review, run it through our attention model embedding layer and attention layer, and look at the weights as a heat map using the show_weights helper function we defined above.

In [ ]:
# RUN THIS. One short review through the trained model, and its attention weights.
X_short, y_short = make_reviews(1, n_words=10, seed=7)
short_words = []
for word_id in X_short[0]:
    short_words.append(vocab[word_id])
print(short_words)

attention_model.eval()
with torch.no_grad():
    vectors = attention_model.embed(X_short)
    new_vectors, weights = attention_model.attention(vectors)
    print("prediction:", attention_model(X_short).argmax(dim=1).item(), "  label:", y_short.item())

show_weights(weights[0], short_words, "Trained attention weights")

Almost every row has all its weight on one column, "hated". Every word in the review learned to look at the feeling word, just like "cold" looked at "pizza" in our by hand example! We never told the network which word mattered. Gradient descent found queries and keys that point at the feeling word, because that is what makes the loss small. (One row, "popcorn", looks at the two "movie" words instead. The average over all 10 words is still mostly "hated", so the prediction is right.)

## What attention does not know - the order of the words

Here is something strange to think about. Attention compares every query with every key, and **nothing in the formula says where a word is in the sentence**. So what happens if we shuffle the words of a review?

In [ ]:
# DEMO ONLY. The same review with the words shuffled.
torch.manual_seed(0)
shuffle = torch.randperm(10)               # a random order for the 10 words
X_shuffled = X_short[:, shuffle]

shuffled_words = []
for word_id in X_shuffled[0]:
    shuffled_words.append(vocab[word_id])
print(shuffled_words)

with torch.no_grad():
    print("scores, in order:  ", attention_model(X_short))
    print("scores, shuffled:  ", attention_model(X_shuffled))

Exactly the same scores! For our made up reviews that is fine, because only the feeling word matters. But for real language this is a big problem: "the dog bit the man" and "the man bit the dog" use the same words. Later in class we will talk about a fix for this!!!

:::{.callout-note icon=false}
## Q5. Write this one out by hand

**a.** In the trained heat map, almost every word puts its weight on "hated". Why is that a good thing to learn for this task?

**b.** Write two sentences that use exactly the same words but mean different things. Would our attention model be able to tell them apart? Why not?
:::

---

# Putting It Together

## What this looks like in real life - The Whole Process in One Cell

Here is everything in one cell, with no helper function. The comments show the places you could change things.

In [ ]:
# LEARN TO WRITE THIS.
# An attention review classifier from start to finish.
# It uses math, vocab_size, X_train_long, y_train_long, X_valid_long, y_valid_long from above.

# 1. The attention layer. Here you could change how the scores are scaled
class SelfAttention(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.query = nn.Linear(d, d, bias=False)     # learned queries
        self.key = nn.Linear(d, d, bias=False)       # learned keys
        self.value = nn.Linear(d, d, bias=False)     # learned values

    def forward(self, X):
        Q = self.query(X)
        K = self.key(X)
        V = self.value(X)
        scores = Q @ K.transpose(1, 2)               # every query dotted with every key
        scores = scores / math.sqrt(K.shape[-1])     # divide by the square root of d
        weights = torch.softmax(scores, dim=-1)      # each row adds up to 1
        return weights @ V, weights                  # the weighted average of the values, and the weights

# 2. The model. Here you could change the embedding size (8, in three places)
class ReviewAttention(nn.Module):
    def __init__(self):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, 8)
        self.attention = SelfAttention(8)
        self.out = nn.Linear(8, 2)

    def forward(self, X):
        vectors = self.embed(X)                          # (reviews, words, 8)
        new_vectors, weights = self.attention(vectors)   # every word looks at every word
        summary = new_vectors.mean(dim=1)                # average over the words: (reviews, 8)
        return self.out(summary)                         # 2 scores per review

torch.manual_seed(42)
model = ReviewAttention()

# 3. The batches, the loss and the optimizer. Here you could change the batch size or the learning rate
train_loader = DataLoader(TensorDataset(X_train_long, y_train_long), batch_size=32, shuffle=True)
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

# 4. The training loop
n_epochs = 5
for epoch in range(n_epochs):
    model.train()                             # training mode
    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()                 # clear the old gradients
        scores = model(X_batch)               # forward pass: 2 scores per review
        loss = loss_fn(scores, y_batch)       # how far off are we?
        loss.backward()                       # backward pass
        optimizer.step()                      # one step of Adam

    # 5. After every epoch, check the validation accuracy
    model.eval()                              # prediction mode
    with torch.no_grad():
        y_pred = model(X_valid_long).argmax(dim=1)
        accuracy = (y_pred == y_valid_long).float().mean().item()
    print("epoch", epoch + 1, "  valid accuracy:", round(accuracy, 4))

Everything from today, in one list:

1. Every word gets a **query** (what it is looking for), a **key** (what it offers) and a **value** (what it hands over).
2. Attention compares every query with every key (dot products), divides by $\sqrt{d}$, turns each row into weights with a softmax, and takes the weighted average of the values: $\text{softmax}(QK^T / \sqrt{d})\,V$.
3. In **self-attention** the queries, keys and values are made from the words' own embeddings with three learned `nn.Linear` layers.
4. Every word is one step from every other word, so there is no long chain for the gradient to vanish through.
5. We can look at the weights as a heat map, to see which words each word looked at.
6. Attention does not know the order of the words. A transformer block fixes that with positional encodings.

# New Commands Today

| Command | What it does | The thing that trips people up |
|---|---|---|
| `Q @ K.T` | every query dotted with every key, for one sentence | for a batch use `K.transpose(1, 2)`, `.T` would also flip the reviews |
| `torch.softmax(scores, dim=-1)` | turns each row of scores into weights that add up to 1 | `dim=-1` means the last dimension, so each row. The wrong `dim` gives columns that add to 1 |
| `weights @ V` | the weighted average of the values, for every word | the number of columns in `weights` has to match the number of rows in `V` (words) |
| `nn.Linear(d, d, bias=False)` | a linear layer with no bias | here it makes the queries, keys and values |
| `tensor.mean(dim=1)` | averages over the words | gives one summary vector per review |
| `F.scaled_dot_product_attention(Q, K, V)` | the whole formula, built in | needs `import torch.nn.functional as F`. It does not give back the weights |
| `plt.imshow(weights, cmap="Blues")` | draws a matrix as a heat map | rows are the words looking, columns the words being looked at |

:::{.callout-tip icon=false}
## You Try: optional code

Nothing here is collected. Work through it if you want the idea to stick.

**1.** In the "cold" example, change the query for "cold" to [4, 4, 0, 0] ("I am looking for a thing that is food"). What happens to the weights? Now try [0, 0, 4, 0] ("I am looking for a describing word"). Which word does "cold" look at now, and why?

**2.** Take out the `scores = scores / math.sqrt(K.shape[-1])` line in the real life cell. Does the model still learn?

**3.** Train `ReviewAttention` on 500 word reviews (make 4,000 of them). How long does an epoch take compared to 100 words? Every word looks at every word, so 5 times the words is 25 times the scores.
:::

# Before Next Class

1. In your lecture notes notebook, add your handwritten notes and answers to the questions.
2. Do the practice problems in `HW_day16.ipynb`. They are practice, not graded, and the solutions are in the same folder.
3. Check the class website for what is due this week.
4. Coming up next: the transformer block, positional encodings and multi-head attention.

# Answers to the Q Boxes

Try every one of these by hand first. These are short summaries, not full answers, and the writing out is the part that does the work.

:::{.callout-note collapse="true"}
## Q1. Attention by hand

**a.** "pizza", with 0.711. "cold" is a describing word, so its query looks for a thing, and "pizza" is the only word whose key says "I am a thing" (dot product 4). "cold" needs to know what is cold, and the answer is the pizza.

**b.** The softmax makes the weights positive and makes them add up to 1, so the output is a weighted average of the values. Bigger scores still get bigger weights. Raw scores could be negative or huge and would not average anything.
:::

:::{.callout-note collapse="true"}
## Q2. Every word at once

**a.** 10 by 10. Row 3 is how much the third word looks at each of the 10 words, and it adds up to 1.

**b.** With more numbers in each key the dot products get bigger. Big scores make the softmax put almost all the weight on one word (20 and 10 gave 0.99995 and 0.00005), so it stops averaging and the gradients get close to 0. Dividing by $\sqrt{d}$ keeps the scores a reasonable size.
:::

:::{.callout-note collapse="true"}
## Q3. Self-attention

**a.** From each word's own embedding, through three `nn.Linear` layers. Nobody picks them, the weights of those layers are learned by gradient descent.

**b.** Three 8 by 8 weight matrices with no bias, $3 \times 64 = 192$. No, the same three layers are used for every word, so the length of the review does not matter (just like the RNN).
:::

:::{.callout-note collapse="true"}
## Q4. The long reviews

**a.** In the RNN, a feeling word near the start has to pass through up to 99 steps of memory to reach the end, and the gradient shrinks at every step back. With attention every word looks at every other word directly, in one step.

**b.** The network can only learn from a word if the gradient for that word is not 0. With attention, a feeling word at the start gets as much gradient as one at the end, so the network learns from it wherever it is.
:::

:::{.callout-note collapse="true"}
## Q5. Looking inside

**a.** The only word that decides the label is the feeling word. If every word pulls in the feeling word's value, then the average over all the words is mostly the feeling word, and the last layer can read positive or negative from it.

**b.** For example "the dog bit the man" and "the man bit the dog". No. Attention compares queries and keys and nothing in it knows where a word is, so the same words in a different order give the same average (we saw the exact same scores for a shuffled review). The fix is positional encodings, in the transformer block.
:::